# 🔬 ImageToChaste: Interactive Quickstart Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/proshanto-c/ImageToChaste/blob/main/notebooks/quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/proshanto-c/ImageToChaste)

This tutorial demonstrates how to take a raw microscopy image of an epithelial cell colony (*Drosophila melanogaster* embryonic germband extension), apply adaptive contrast pre-processing, segment individual cell bodies using **Meta's Segment Anything Model 2 (SAM 2)**, and export topologically verified simulation meshes for Oxford's **Chaste** C++ simulation framework.

## 1. Setup & Installation
If running in Google Colab, execute this cell to install dependencies and `imagetochaste`.

In [ ]:
# If running in Colab, install SAM 2 and ImageToChaste
import sys

if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/facebookresearch/sam2.git
    !pip install -q git+https://github.com/proshanto-c/ImageToChaste.git
    !mkdir -p data/sample
    !wget -q -O data/sample/drosophila_germband_f009.png https://raw.githubusercontent.com/proshanto-c/ImageToChaste/main/data/sample/drosophila_germband_f009.png

from pathlib import Path

import matplotlib.pyplot as plt
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if hasattr(torch.backends, 'mps'):
    print(f"Apple Silicon MPS Available: {torch.backends.mps.is_available()}")

## 2. Download SAM 2 Checkpoint
We download Meta's official SAM 2 checkpoint. For this quickstart tutorial, we use `tiny` for rapid inference, but `large` is recommended for production accuracy.

In [ ]:
from imagetochaste.download_weights import download_checkpoint

# Download tiny checkpoint (~75 MB) for instant interactive inference
checkpoint_path = download_checkpoint(model_type="tiny", output_dir="checkpoints")
print(f"Checkpoint ready at: {checkpoint_path}")

## 3. Load & Pre-process Microscopy Scan
Microscopy images often suffer from uneven illumination and faint cell membrane boundaries. `ImageToChaste` applies **Contrast Limited Adaptive Histogram Equalization (CLAHE)** followed by Gaussian background subtraction to homogenize the image field.

In [ ]:
from imagetochaste.segmentation.preprocessor import load_image, preprocess_microscopy_image

image_path = "data/sample/drosophila_germband_f009.png"
raw_img = load_image(image_path)
enhanced_img = preprocess_microscopy_image(raw_img, clahe_clip_limit=3.0)

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(raw_img)
axes[0].set_title("Raw Microscopy Scan (Frame 009)", fontsize=14)
axes[0].axis("off")

axes[1].imshow(enhanced_img)
axes[1].set_title("CLAHE + Background Subtracted", fontsize=14)
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Run SAM 2 Cell Segmentation & Statistical Outlier Rejection
We initialize the `SAMAdapter` and run automatic cell segmentation with two-stage statistical area filtering to strip away non-cellular debris and over-segmented tiles.

In [ ]:
from imagetochaste import SAMAdapter
from imagetochaste.segmentation.utils import create_mask_overlay

# Initialize adapter (automatically picks CUDA, MPS, or CPU)
adapter = SAMAdapter(checkpoint=checkpoint_path, device="auto")

# Run segmentation
print("Segmenting cell colony...")
masks, stats = adapter.generate_masks(enhanced_img, filter_area=True)

print(f"Initial masks discovered: {stats.get('initial_count', len(masks))}")
print(f"Final filtered cell masks: {len(masks)}")

# Visualize segmentation overlay
overlay = create_mask_overlay(raw_img, masks, alpha=0.5, draw_borders=True)

plt.figure(figsize=(10, 8))
plt.imshow(overlay)
plt.title(f"SAM 2 Segmented Cells (n={len(masks)})", fontsize=16)
plt.axis("off")
plt.show()

## 5. Geometric Extraction & Bounded Voronoi Meshing
From each cell mask, we compute the center-of-mass centroid and generate a bounded Voronoi diagram that mirrors boundaries across the bounding box to produce closed, valid polygons.

In [ ]:
from imagetochaste import build_voronoi_mesh, compute_centroids_from_masks

# 1. Extract centroids
centroids = compute_centroids_from_masks(masks)
print(f"Computed {len(centroids)} cell centroids.")

# 2. Build Voronoi mesh for Chaste
h, w, _ = raw_img.shape
mesh = build_voronoi_mesh(centroids, bounding_box=(0, 0, w, h))
print(f"Constructed mesh with {mesh.num_nodes} junction nodes and {mesh.num_elements} polygonal cells.")

# Plot Voronoi mesh wireframe over the original scan
plt.figure(figsize=(10, 8))
plt.imshow(raw_img)

# Plot centroids
cx = [c[0] for c in centroids]
cy = [c[1] for c in centroids]
plt.scatter(cx, cy, color='cyan', s=12, label='Cell Centroids', zorder=4)

# Plot element polygons
node_dict = {n.node_id: (n.x, n.y) for n in mesh.nodes}
for elem in mesh.elements:
    poly_coords = [node_dict[nid] for nid in elem.node_ids] + [node_dict[elem.node_ids[0]]]
    px, py = zip(*poly_coords)
    plt.plot(px, py, color='lime', linewidth=1.0, alpha=0.8)

plt.title("Chaste Voronoi Mesh Overlaid on Microscopy Scan", fontsize=16)
plt.legend(loc='upper right')
plt.axis("off")
plt.show()

## 6. Export to Chaste Simulation Formats
Now we export the geometry into exact Chaste ASCII format:
- `.nodes`: Node index, $(x, y)$ coordinates, and boundary marker flags ($0$ for interior, $1$ for boundary).
- `.elements`: Element index, vertex count, and counter-clockwise ordered node IDs.

In [ ]:
from imagetochaste import export_chaste_nodes, export_chaste_vertex_mesh

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# 1. Export NodesOnlyMesh (for NodeBasedCellPopulation)
nodes_path = export_chaste_nodes(centroids, out_dir / "cells.nodes")
print(f"Exported: {nodes_path}")

# 2. Export VertexMesh (for VertexBasedCellPopulation)
node_f, elem_f = export_chaste_vertex_mesh(mesh, out_dir / "vertex_mesh")
print(f"Exported: {node_f} and {elem_f}")

# Preview first 10 lines of each file
print("\n--- First 10 lines of vertex_mesh.nodes ---")
print("\n".join(node_f.read_text().splitlines()[:10]))

print("\n--- First 10 lines of vertex_mesh.elements ---")
print("\n".join(elem_f.read_text().splitlines()[:10]))

## 7. How Chaste Reads These Files in C++

Here is how you ingest the exported mesh in a Chaste simulation:

```cpp
#include "VertexMeshReader.hpp"
#include "MutableVertexMesh.hpp"
#include "VertexBasedCellPopulation.hpp"
#include "OffLatticeSimulation.hpp"
#include "NagaiHondaDifferentialAdhesionForce.hpp"

void RunSimulationFromScan()
{
    // 1. Read files exported by ImageToChaste
    VertexMeshReader<2, 2> mesh_reader("outputs/vertex_mesh");
    MutableVertexMesh<2, 2> cell_mesh;
    cell_mesh.ConstructFromMeshReader(mesh_reader);

    // 2. Initialise cell population
    std::vector<CellPtr> cells;
    GenerateCells(cells, cell_mesh.GetNumElements());
    VertexBasedCellPopulation<2> cell_population(cell_mesh, cells);

    // 3. Configure Off-Lattice Simulation
    OffLatticeSimulation<2> simulator(cell_population);
    simulator.SetOutputDirectory("ChasteMicroscopySim");
    simulator.SetEndTime(24.0); // 24 hours simulation time

    MAKE_PTR(NagaiHondaDifferentialAdhesionForce<2>, p_force);
    simulator.AddForce(p_force);

    simulator.Solve();
}
```